# F21MP Full Pipeline Notebook (All Phases)

This is a single notebook to run and review the full implementation end-to-end.

## Covered workflow
1. Phase 1: Train baseline (A1) and full model (A4)
2. Phase 2: Cross-dataset validation + threshold sweep + adaptation extensions
3. Phase 2 robustness: multi-seed stability + confidence intervals + error analysis
4. Phase 3: Ablation (A1-A4) + hate-speech benchmark
5. Phase 4 (optional): OLID benchmark

> Existing phase-wise notebooks are still available and unchanged.


## Run instructions

- Start Jupyter from project root with `PYTHONPATH=.`
- Use the same kernel for all cells
- Run top-to-bottom the first time
- Re-run only specific phase cells when needed


## 0) Setup and environment checks


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Cannot find F21MP project root.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)
print('Python:', sys.executable)


In [ ]:
# Optional quick dependency check
import importlib
for pkg in ['pandas', 'numpy', 'sklearn', 'xgboost', 'matplotlib', 'joblib']:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, '__version__', 'unknown')
    print(f'{pkg}: {ver}')


---
## 1) Phase 1 - Main model training (Towards Comprehensive)

### Objective
Train and compare:
- **A1**: text-only baseline
- **A4**: text + peerness + repetition

### Outputs generated
- `outputs/models/baseline_a1.joblib`
- `outputs/models/full_a4.joblib`
- `outputs/results/phase1_test_metrics.json`
- confusion matrices under `outputs/media/`


In [ ]:
from src.phase1_training.models.train import train_phase1

phase1 = train_phase1(test_size=0.2, random_state=42, save_outputs=True, verbose=True)


In [ ]:
phase1_summary = pd.DataFrame([
    {
        'model': 'A1_text_only',
        **{k: phase1['baseline_metrics'][k] for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    },
    {
        'model': 'A4_full',
        **{k: phase1['full_metrics'][k] for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    },
])
phase1_summary


### Phase 1 interpretation notes

- Use this section to report the core in-domain comparison (A1 vs A4).
- This result is the main evidence for your research question on feature contribution.


### Phase 1 model artifacts (important)

Phase 1 creates and saves the trained models:

- `outputs/models/baseline_a1.joblib` (A1: text-only baseline)
- `outputs/models/full_a4.joblib` (A4: text + peerness + repetition)

These are then reused in later phases:

- **Phase 2** uses Phase 1 models for zero-shot validation and comparison.
- **Phase 4** reuses `baseline_a1.joblib` for the OLID zero-shot check.


---
## 2) Phase 2 - Cross-dataset validation and adaptation

### Objective
Evaluate transfer to external Twitter/YouTube subsets and test practical improvements:
- zero-shot strategies
- threshold sweep
- domain-adaptive fine-tune
- embedding-style extension


In [ ]:
from src.phase2_validation.evaluation.validation import (
    run_validation,
    sweep_thresholds,
    run_domain_adaptive_finetune,
    run_embedding_extension,
)

val_text = run_validation(strategy='text_only', threshold=0.5, save_outputs=True)
val_full = run_validation(strategy='zero_shot_full', threshold=0.5, save_outputs=True)

display(val_text['summary'])
display(val_full['summary'])


In [ ]:
threshold_sweep = sweep_thresholds(
    strategy='text_only',
    thresholds=[0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.50],
    save_outputs=True,
)
threshold_sweep.head(12)


In [ ]:
best_thresholds = (
    threshold_sweep.sort_values(['subset', 'f1'], ascending=[True, False])
    .groupby('subset', as_index=False)
    .first()[['subset', 'threshold', 'f1', 'precision', 'recall']]
)
best_thresholds


In [ ]:
adapt = run_domain_adaptive_finetune(
    subset_name='twitter_parsed_dataset.csv',
    adapt_fraction=0.20,
    random_state=42,
    save_outputs=True,
)
embed = run_embedding_extension(
    subset_name='twitter_parsed_dataset.csv',
    adapt_fraction=0.20,
    random_state=42,
    save_outputs=True,
)

phase2_ext = pd.DataFrame([
    {'experiment': 'domain_adaptive_finetune', 'threshold': adapt['threshold_selected_on_calib'], **adapt['test_metrics']},
    {'experiment': 'embedding_extension', 'threshold': embed['threshold_selected_on_calib'], **embed['test_metrics']},
])
phase2_ext


### Phase 2 interpretation notes

- Compare zero-shot vs adaptation results.
- Highlight that adaptation recovers useful performance under domain shift.
- Keep this section aligned with dissertation wording.


---
## 3) Phase 2 robustness, confidence intervals, and error analysis

### Objective
Strengthen credibility of adaptation findings with:
- 3-seed robustness check
- variance and 95% CI reporting for F1
- qualitative misclassification examples


In [ ]:
from src.phase2_validation.evaluation.validation import (
    run_adaptation_robustness,
    run_misclassification_analysis,
)

robust = run_adaptation_robustness(
    subset_name='twitter_parsed_dataset.csv',
    adapt_fraction=0.20,
    seeds=(13, 42, 77),
    save_outputs=True,
)

display(robust['per_seed'])


In [ ]:
robust['summary']


In [ ]:
err = run_misclassification_analysis(
    subset_name='twitter_parsed_dataset.csv',
    experiment='embedding_extension',
    adapt_fraction=0.20,
    random_state=42,
    top_n=20,
    save_outputs=True,
)
print('Selected threshold:', round(err['threshold'], 4))
print('Total test errors:', err['n_errors_total'])
err['errors'].head(20)


### Error-analysis discussion prompts

- Which errors are mostly FP vs FN?
- Are there repeated lexical patterns in high-confidence mistakes?
- Does context ambiguity explain false positives?


---
## 4) Phase 3 - Ablation and hate-speech benchmark

### Objective
- Run A1-A4 ablation on Towards Comprehensive
- Run 3-class benchmark on hate-speech dataset


In [ ]:
from src.phase3_ablation_benchmark.evaluation.ablation import run_ablation, run_hate_speech_benchmark

ablation = run_ablation(test_size=0.2, random_state=42, save_outputs=True)
hate = run_hate_speech_benchmark(test_size=0.2, random_state=42, save_outputs=True)

print('Ablation summary')
display(ablation['summary'])
print('Hate-speech benchmark summary')
display(hate['summary'])


### Phase 3 interpretation notes

- Use ablation results to quantify feature contribution.
- Use 3-class benchmark to show external text-only baseline behavior.


---
## 5) Phase 4 (optional) - OLID benchmark

### Objective
Compare:
- zero-shot Phase 1 A1 on OLID
- in-domain OLID text-only training


In [ ]:
RUN_PHASE4 = True  # set False to skip this optional phase

if RUN_PHASE4:
    from src.phase4_olid.data.load_olid import run_olid_benchmark
    olid = run_olid_benchmark(save_outputs=True)
    display(olid['summary'])
else:
    print('Phase 4 skipped.')


### Phase 4 interpretation notes

- If in-domain OLID >> zero-shot OLID, this supports domain-shift observations from Phase 2.


---
## 6) Final combined summary table


In [ ]:
rows = []

if 'phase1_summary' in globals():
    for _, r in phase1_summary.iterrows():
        rows.append({'phase': 'phase1', 'experiment': r['model'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

if 'phase2_ext' in globals():
    for _, r in phase2_ext.iterrows():
        rows.append({'phase': 'phase2_extension', 'experiment': r['experiment'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

if 'ablation' in globals():
    for _, r in ablation['summary'].iterrows():
        rows.append({'phase': 'phase3_ablation', 'experiment': r['experiment'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

if 'hate' in globals():
    for _, r in hate['summary'].iterrows():
        rows.append({'phase': 'phase3_hate_speech', 'experiment': r['model'], 'f1': r.get('macro_f1'), 'roc_auc': np.nan})

if 'olid' in globals() and RUN_PHASE4:
    for _, r in olid['summary'].iterrows():
        rows.append({'phase': 'phase4_olid', 'experiment': r['model'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

final_summary = pd.DataFrame(rows)
final_summary


## 7) Output files generated

Typical generated files are saved under:
- `outputs/models/`
- `outputs/results/`
- `outputs/media/`

Use these artifacts directly in your report/dissertation tables and figures.
